In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import os, time

def scrape(path, headless=False):
    chrome_options = Options()
    if headless:
        chrome_options.add_argument("--headless")
    chrome_options.add_argument("--window-size=1920x1080")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--hide-scrollbars")
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    driver.get(path)
    driver.maximize_window()
    
    # Login
    try:
        user_name = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, 'username')))
        user_name.send_keys("fourbrotherstrading@icloud.com")
    except:
        print("No username field")

    try:
        password = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, 'password')))
        driver.execute_script("arguments[0].scrollIntoView();", password)
        password.send_keys("Sultanmirza1501#")
    except:
        print("No password field")

    try:
        sign_in_btn = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, './/button[text()="Sign in"]')))
        sign_in_btn.click()
    except:
        print("No sign-in button")

    try:
        popup = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, 'start-btn')))
        popup.click()
    except:
        print("No popup found")

    return driver


url = "https://www.cityauctiongroup.com/portal/auction/buyer/webauction/auction/2038"
driver = scrape(url, headless=False)

os.makedirs("live_html", exist_ok=True)
os.makedirs("screenshots", exist_ok=True)

previous_title = ""
file_counter = 1  

try:
    while True:
        if not driver.window_handles:
            print("Browser closed. Exiting.")
            break

        try:
            car_title = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.XPATH, './/div[@class="col-9"]/h1'))
            )
            current_title = car_title.text.strip()

            if current_title != previous_title:
                previous_title = current_title
                print(f"New vehicle: {current_title}")

                try:
                    reg_element = driver.find_element(
                        By.XPATH, 
                        '//table[contains(@class,"vehicle-table")]//th[text()="Registration"]/following-sibling::td'
                    )
                    reg_number = reg_element.text.strip().replace(" ", "_")
                except:
                    reg_number = current_title[:10].replace(" ", "_")  
                html_file = os.path.join("live_html", f"{file_counter}_{reg_number}.html")
                screenshot_file = os.path.join("screenshots", f"{file_counter}_ref.png")

              
                with open(html_file, "w", encoding="utf-8") as f:
                    f.write(driver.page_source)
                print(f"Saved HTML: {html_file}")

                
                driver.save_screenshot(screenshot_file)
                print(f"Saved Screenshot: {screenshot_file}")

                file_counter += 1  

            time.sleep(2)

        except Exception as e:
            print("Error or no more cars:", e)
            time.sleep(2)

except KeyboardInterrupt:
    print("Stopped by user.")
finally:
    driver.quit()
    print("Browser closed.")


New vehicle: Lot 1: Mini Countryman
Saved HTML: live_html\1_AJ74EVT.html
Saved Screenshot: screenshots\1_ref.png
New vehicle: Lot 2: Jeep Avenger
Saved HTML: live_html\2_LD25ZZW.html
Saved Screenshot: screenshots\2_ref.png
New vehicle: Lot 3: Hyundai Tucson
Saved HTML: live_html\3_BYZ6377.html
Saved Screenshot: screenshots\3_ref.png
New vehicle: Lot 4: Kia Sportage
Saved HTML: live_html\4_ISZ6266.html
Saved Screenshot: screenshots\4_ref.png
New vehicle: Lot 5: Ford Focus
Saved HTML: live_html\5_AF72YFL.html
Saved Screenshot: screenshots\5_ref.png
New vehicle: Lot 6: Kgm Korando
Saved HTML: live_html\6_YK24XWH.html
Saved Screenshot: screenshots\6_ref.png
New vehicle: Lot 7: Volkswagen Polo
Saved HTML: live_html\7_YJ74RHR.html
Saved Screenshot: screenshots\7_ref.png
New vehicle: Lot 8: Kia Picanto
Saved HTML: live_html\8_LY25YSU.html
Saved Screenshot: screenshots\8_ref.png
New vehicle: Lot 9: Nissan Qashqai
Saved HTML: live_html\9_DU72BNL.html
Saved Screenshot: screenshots\9_ref.png
New 

InvalidSessionIdException: Message: invalid session id: session deleted as the browser has closed the connection
from disconnected: not connected to DevTools
  (Session info: chrome=143.0.7499.170); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x321213
	0x321254
	0x10e6dd
	0xfc4cc
	0xfd456
	0x10f167
	0xec5bb
	0x18afd1
	0x17b4b3
	0x14d321
	0x14e1d4
	0x575254
	0x57080b
	0x58d0ea
	0x33b118
	0x34311d
	0x329518
	0x3296d9
	0x313a68
	0x76d45d49
	0x77b1d5db
	0x77b1d561
	0


In [2]:
import os
import csv
from bs4 import BeautifulSoup

def parse_last_html():
    folder = "live_html"
    files = sorted(os.listdir(folder), key=lambda x: os.path.getmtime(os.path.join(folder, x)))
    last_file = os.path.join(folder, files[-1])
    
    print(f"📌 Reading File: {last_file}")
    
    with open(last_file, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")

    bid_list = soup.find("ul", id="biddinghistory")
    if not bid_list:
        print("⚠️ No Bidding History Found!")
        return

    items = bid_list.find_all("li")

    results = []
    current_lot = None
    bids = []
    status = ""

    for li in items:
        text = li.get_text(strip=True).lower()
        raw = li.get_text(strip=True)

        if "lot changed:" in text:
            if current_lot:
                last_bid = bids[0] if bids else ""
                results.append([current_lot, bids, status, last_bid])
            current_lot = raw.replace("Lot changed:", "").strip()
            bids = []
            status = ""

        elif "not sold" in text:
            status = "not_sold"

        elif "provisionally" in text and "sold" in text:
            status = "provisionally"
            price = raw.split("for")[-1].strip()
            bids.insert(0, price)

        elif "sold" in text and "not sold" not in text:
            status = "sold"
            price = raw.split("for")[-1].strip()
            bids.insert(0, price)

        elif "progress" in text:
            status = "in_progress"

        elif "bid:" in text:
            if status in ["sold", "provisionally"]:
                price = raw.split("£")[-1].strip()
                bids.append("£" + price)

    # Last lot
    if current_lot:
        last_bid = bids[0] if bids else ""
        results.append([current_lot, bids, status, last_bid])

    # Merge empty lots into previous
    cleaned_results = []
    for lot, bids_list, st, last_bid in results:
        if bids_list or st in ["sold", "provisionally"]:  # keep if it has bids or sold/provisional
            cleaned_results.append([lot, bids_list, st, last_bid])
        else:
            # merge status if previous exists
            if cleaned_results:
                cleaned_results[-1][2] = st  # update status of previous lot

    # Flatten bids list to string
    final_results = []
    for lot, bids_list, st, last_bid in cleaned_results:
        bids_str = ", ".join(bids_list) if bids_list else ""
        final_results.append([lot, bids_str, st, last_bid])

    # Print output
    print("\n=== BIDDING RESULT ===")
    for row in final_results:
        print(row)

    # Save CSV
    csv_file = "CCA_LIVE_data.csv"
    with open(csv_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["Lot", "Bids", "Bidding Status", "Last Bid"])
        for row in final_results:
            writer.writerow(row)

    print(f"\n💾 CSV Saved Successfully: {csv_file}")


# RUN
parse_last_html()


📌 Reading File: live_html\159_WO73VTU.html

=== BIDDING RESULT ===
['157', '£14,400, £14,400, £14,300, £14,200, £14,100', 'provisionally', '£14,400']
['156', '£16,400, £16,400, £16,300, £16,200, £16,100, £16,000', 'sold', '£16,400']
['155', '£14,400, £14,400, £14,300, £14,200', 'provisionally', '£14,400']
['154', '£31,500, £31,500, £31,400, £31,300, £31,200, £31,100', 'not_sold', '£31,500']
['152', '£17,200, £17,200, £17,100, £17,000', 'sold', '£17,200']
['151', '£18,700, £18,700, £18,600, £18,500', 'provisionally', '£18,700']
['150', '£18,200, £18,200, £18,100, £18,000', 'sold', '£18,200']
['149', '£25,200, £25,200, £25,100, £25,000', 'provisionally', '£25,200']
['148', '£17,200, £17,200, £17,100, £17,000', 'sold', '£17,200']
['147', '£13,600, £13,600, £13,500, £13,400', 'provisionally', '£13,600']
['146', '£7,600, £7,600, £7,500', 'sold', '£7,600']
['145', '£13,800, £13,800, £13,700, £13,600', 'provisionally', '£13,800']
['144', '£18,100, £18,100, £18,000', 'sold', '£18,100']
['143',

In [3]:
import os
import csv

def save_lot_reg_csv():
    folder = "live_html"
    files = sorted(os.listdir(folder), key=lambda x: int(x.split("_")[0])) 

    csv_file = "lot_reg_mapping.csv"
    with open(csv_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["Lot Number", "Reg"])  # header

        for file in files:
            lot_number = file.split("_")[0]
            reg_name = file.split("_")[1].replace(".html", "")
            writer.writerow([lot_number, reg_name])

    print(f"\n💾 CSV Saved Successfully: {csv_file}")

# RUN
save_lot_reg_csv()



💾 CSV Saved Successfully: lot_reg_mapping.csv


In [4]:
import pandas as pd
import os


live_df = pd.read_csv("CCA_LIVE_data.csv")
reg_df = pd.read_csv("lot_reg_mapping.csv")


reg_df["Lot Number"] = reg_df["Lot Number"].astype(int)
live_df = live_df.reset_index(drop=True)


live_df["Reg"] = reg_df["Reg"].tolist()[:len(live_df)]  


final_csv = "CCg_LIVE_data_final.csv"
live_df.to_csv(final_csv, index=False)

os.remove("CCA_LIVE_data.csv")
os.remove("lot_reg_mapping.csv")

print(f"💾 Merged CSV saved as {final_csv} and original files deleted.")


💾 Merged CSV saved as CCg_LIVE_data_final.csv and original files deleted.


In [5]:
import pandas as pd

# Load CSVs
live_df = pd.read_csv("CCg_LIVE_data_final.csv")
reg_df = pd.read_csv("cag_data.csv")

# Ensure Reg columns are string type
live_df['Reg'] = live_df['Reg'].astype(str)
reg_df['Reg'] = reg_df['Reg'].astype(str)

# Keep only relevant columns from live_df and rename Bids -> Bidding History
live_df = live_df[["Bids", "Bidding Status", "Last Bid", "Reg"]].rename(columns={"Bids": "Bidding History"})

# Merge on Reg, keep all rows from reg_df
merged_df = pd.merge(reg_df, live_df, on="Reg", how="left")  # merge live data at the end

# Remove duplicates if any
merged_df = merged_df.drop_duplicates(subset="Reg")

# Save final CSV
merged_df.to_csv("final_cag.csv", index=False)

print(f"💾 final_cag.csv created successfully! Total records: {len(merged_df)}")


💾 final_cag.csv created successfully! Total records: 153
